# Lesson 3.3 — experiments: evaluation functions and depth limits

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In `"reference"` mode the weighted-lines evaluation stands in for your evaluation function.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

The function `run` runs one search and returns its result, the number of positions visited and the time in seconds. The function `player(depth, evaluate)` creates a player that chooses moves with your search at the given depth and with the given evaluation. The opponent in the tournament is the reference search at depth 2 with an evaluation that always returns 0, as in the match test. The open-lines and weighted-lines evaluations come from the course's reference version.

In [ ]:
import time

import matplotlib.pyplot as plt

from introai.games import KInARow, reference
from introai.games.players import match, search_player
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'
BOARD = KInARow(5, 5, 4)
if IMPLEMENTATION == "mine":
    your_evaluate = algorithms.evaluate
else:
    your_evaluate = reference.weighted_lines


def run(search, game, state, *args, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state, *args)
    except NotImplementedError as error:
        raise RuntimeError(str(error)) from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


def player(depth, evaluate):
    """A player that chooses moves with your depth-limited search."""

    def choose(game, state):
        (_, move), _, _ = run(algorithms.h_alpha_beta_search, game, state, depth, evaluate)
        return move

    return choose


opponent = search_player(reference.h_alpha_beta_search, depth=2, evaluate=reference.zero_evaluation)

## Experiment 1: a tournament on the 5 × 5 board

Four evaluation functions, each at depths 1, 2 and 3, play the same match as in the test: 12 games from 6 openings, each with crosses and with noughts, against a depth-2 search without an evaluation. The table shows wins, draws and losses. The computation takes a few seconds.

**Your prediction.** Which evaluation will do best? Will the zero evaluation at depth 3 beat the opponent at depth 2? Does more depth always help?

*My prediction:* …

In [ ]:
evaluations = [
    ("zero", reference.zero_evaluation),
    ("open lines", reference.open_lines),
    ("weighted lines", reference.weighted_lines),
    ("yours", your_evaluate),
]
start = time.perf_counter()
print(f"{'evaluation':<16}{'depth 1':>10}{'depth 2':>10}{'depth 3':>10}   (wins/draws/losses)")
for name, evaluate in evaluations:
    row = []
    for depth in (1, 2, 3):
        table = match(BOARD, player(depth, evaluate), opponent)
        row.append(f"{table['win']}/{table['draw']}/{table['loss']}")
    print(f"{name:<16}{row[0]:>10}{row[1]:>10}{row[2]:>10}")
print(f"\n{time.perf_counter() - start:.1f} s")

**Think about it.** How did the zero evaluation do at depth 1, and at depth 3? What does that say about whether a good evaluation or a deeper search matters more? Does your evaluation also win at depths 1 and 3, or only at depth 2, which the test checks?

## Experiment 2: a position with a threat

X is to move. O has two noughts in the first column, and the cells above and below them are empty. The search chooses a move at depths 1 to 4, once with the weighted-lines evaluation and once with yours.

**Your prediction.** Which move should X play, and why? From what depth on will the search find it?

*My prediction:* …

In [ ]:
threat = BOARD.from_rows([".....", "O....", "O..X.", ".X...", "....."])
print(BOARD.render(threat))
print()
print(f"{'depth':<7}{'weighted lines':>24}{'yours':>24}")
for depth in (1, 2, 3, 4):
    row = []
    for evaluate in (reference.weighted_lines, your_evaluate):
        (value, move), _, _ = run(algorithms.h_alpha_beta_search, BOARD, threat, depth, evaluate)
        row.append(f"{move!s:>10} {value:+.3f}")
    print(f"{depth:<7}{row[0]:>24}{row[1]:>24}")

**Think about it.** Why does the depth-1 search not see the danger? Look at the scores too: why are they clearly lower at even depths than at odd ones? Who makes the last move before the horizon in each case?

## Experiment 3: the cost of depth

Your search with your evaluation looks for the first move on the empty 5 × 5 board with depths 1 to 4. We measure how many positions it visits and how long this one move takes. Both vertical axes have a logarithmic scale.

**Your prediction.** How many times more positions does each extra ply cost? How long would a move at depth 8 take?

*My prediction:* …

In [ ]:
depths = [1, 2, 3, 4]
positions, seconds = [], []
for depth in depths:
    _, visited, elapsed = run(
        algorithms.h_alpha_beta_search, BOARD, BOARD.initial, depth, your_evaluate
    )
    positions.append(visited)
    seconds.append(elapsed)
    print(f"depth {depth}: {visited:>7} positions, {elapsed:.3f} s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in (
    (axes[0], positions, "Positions visited for one move (log scale)"),
    (axes[1], seconds, "Seconds for one move (log scale)"),
):
    ax.plot(
        depths,
        values,
        color="#2a78d6",
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
    )
    ax.set_yscale("log")
    ax.set_xticks(depths)
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("search depth (plies)", color="#52514e")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** What is the ratio of the numbers of positions between successive depths? Compare it with the number of moves on the board (25, then 24, …) and with its square root. How deep could your program search in one second?